# 🧠 Materi Kuliah Deep Learning
## Pertemuan 4: Recurrent Neural Networks (RNN) dan LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hendrick02121977/Deep-Learning/blob/main/notebooks/04_RNN_dan_LSTM.ipynb)

---

**Tujuan Pembelajaran:**
- Memahami konsep Recurrent Neural Networks (RNN)
- Memahami masalah Vanishing Gradient dan solusinya
- Mengenal arsitektur LSTM dan GRU
- Mengimplementasikan LSTM untuk prediksi time series
- Mengimplementasikan analisis sentimen dengan LSTM

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print("Library berhasil diimport!")

## 1. Mengapa RNN?

### Data Sekuensial
Banyak data memiliki **urutan waktu** atau **dependensi konteks** yang penting:
- **Teks**: "Bank tepi sungai" vs "Bank tempat menyimpan uang" - makna bergantung konteks
- **Time series**: Prediksi harga saham, cuaca, sensor
- **Audio**: Pengenalan suara dan musik
- **Video**: Sequence frame gambar

### Keterbatasan Neural Network Biasa
Feedforward NN tidak bisa "mengingat" informasi sebelumnya karena setiap input diproses secara independen.

### Solusi: RNN
RNN memiliki **koneksi berulang** yang memungkinkan informasi mengalir dari satu langkah waktu ke langkah berikutnya.

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

In [ ]:
# Implementasi RNN cell sederhana dari scratch
class SimpleRNNCell:
    def __init__(self, input_size, hidden_size):
        # Parameter
        self.Wx = np.random.randn(input_size, hidden_size) * 0.01  # Input -> Hidden
        self.Wh = np.random.randn(hidden_size, hidden_size) * 0.01 # Hidden -> Hidden
        self.b  = np.zeros(hidden_size)                             # Bias
    
    def forward(self, x_t, h_prev):
        """
        x_t   : input pada timestep t
        h_prev: hidden state dari timestep t-1
        """
        h_t = np.tanh(np.dot(x_t, self.Wx) + np.dot(h_prev, self.Wh) + self.b)
        return h_t

# Demo: Proses sequence
rnn_cell = SimpleRNNCell(input_size=3, hidden_size=4)

# Sequence input: 5 timestep, setiap timestep 3 fitur
sequence = np.random.randn(5, 3)

print("Demo RNN - Memproses Sequence")
print("=" * 40)

h = np.zeros(4)  # Initial hidden state = 0
print(f"h_0 (initial): {h.round(4)}\n")

for t, x_t in enumerate(sequence):
    h = rnn_cell.forward(x_t, h)
    print(f"t={t+1} | input: {x_t.round(3)} | hidden: {h.round(4)}")

print(f"\nFinal hidden state (h_5): {h.round(4)}")
print("\nHidden state ini mengandung 'memori' dari seluruh sequence!")

## 2. Masalah Vanishing Gradient

### Masalah pada RNN Standar
Saat training RNN dengan backpropagation through time (BPTT), gradien harus melewati banyak langkah waktu:

$$\frac{\partial \mathcal{L}}{\partial h_0} = \frac{\partial \mathcal{L}}{\partial h_T} \cdot \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

Jika setiap term $< 1$, hasil kali akan mendekati 0 → **Vanishing Gradient**

### Solusi: LSTM (Long Short-Term Memory)

In [ ]:
# Visualisasi masalah vanishing gradient
def compute_gradient_through_time(T, weight=0.9):
    """Simulasi gradient melalui T timestep dengan bobot tertentu"""
    gradients = [weight**t for t in range(T)]
    return gradients

T = 50  # Sequence panjang 50

gradients_vanishing = compute_gradient_through_time(T, weight=0.9)
gradients_exploding = compute_gradient_through_time(T, weight=1.1)
gradients_stable = compute_gradient_through_time(T, weight=1.0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Masalah Vanishing & Exploding Gradient dalam RNN', fontsize=13, fontweight='bold')

timesteps = list(range(T))

axes[0].semilogy(timesteps, gradients_vanishing, 'b-o', markersize=3)
axes[0].set_title('Vanishing Gradient\n(weight = 0.9 per step)')
axes[0].set_xlabel('Jarak dari timestep akhir')
axes[0].set_ylabel('Besar Gradien (log scale)')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0.01, color='red', linestyle='--', alpha=0.5, label='Hampir nol')
axes[0].legend()

axes[1].semilogy(timesteps, gradients_stable, 'g-o', markersize=3)
axes[1].set_title('Gradien Stabil\n(weight = 1.0 per step)')
axes[1].set_xlabel('Jarak dari timestep akhir')
axes[1].set_ylabel('Besar Gradien (log scale)')
axes[1].grid(True, alpha=0.3)

axes[2].semilogy(timesteps, gradients_exploding, 'r-o', markersize=3)
axes[2].set_title('Exploding Gradient\n(weight = 1.1 per step)')
axes[2].set_xlabel('Jarak dari timestep akhir')
axes[2].set_ylabel('Besar Gradien (log scale)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Vanishing - Gradien pada t=0: {gradients_vanishing[0]:.4f}")
print(f"Vanishing - Gradien pada t=50: {gradients_vanishing[-1]:.6f}")
print("\nRNN 'lupa' konteks dari awal sequence!")

## 3. LSTM - Long Short-Term Memory

LSTM memiliki **gating mechanism** yang mengontrol aliran informasi:

### Gate LSTM:
- **Forget Gate** ($f_t$): Memutuskan informasi apa yang dilupakan dari cell state
  $$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$

- **Input Gate** ($i_t$): Memutuskan informasi baru apa yang disimpan
  $$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$
  $$\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$$

- **Cell State Update**:
  $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

- **Output Gate** ($o_t$): Memutuskan output berdasarkan cell state
  $$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$$
  $$h_t = o_t \odot \tanh(C_t)$$

## 4. Aplikasi 1: Prediksi Time Series dengan LSTM

In [ ]:
# Buat dataset time series sinusoidal dengan noise
np.random.seed(42)
t = np.linspace(0, 10 * np.pi, 1000)
series = np.sin(t) + 0.5 * np.sin(3 * t) + 0.2 * np.random.randn(1000)

plt.figure(figsize=(14, 4))
plt.plot(t[:200], series[:200], 'b-', linewidth=1.5, label='Time Series')
plt.title('Contoh Data Time Series (200 titik pertama)', fontsize=12, fontweight='bold')
plt.xlabel('Waktu')
plt.ylabel('Nilai')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

def create_sequences(data, seq_length):
    """Buat sequences X (input) dan y (target) dari time series"""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

# Normalisasi
from sklearn.preprocessing import MinMaxScaler
scaler_ts = MinMaxScaler()
series_scaled = scaler_ts.fit_transform(series.reshape(-1, 1)).flatten()

# Buat sequences
SEQ_LEN = 30  # Gunakan 30 titik sebelumnya untuk prediksi
X_seq, y_seq = create_sequences(series_scaled, SEQ_LEN)
X_seq = X_seq.reshape(X_seq.shape[0], X_seq.shape[1], 1)  # Tambah dimensi fitur

# Split train/test
split = int(0.8 * len(X_seq))
X_train_ts, X_test_ts = X_seq[:split], X_seq[split:]
y_train_ts, y_test_ts = y_seq[:split], y_seq[split:]

print(f"Training samples: {X_train_ts.shape}")
print(f"Test samples    : {X_test_ts.shape}")
print(f"Input shape     : (batch, timesteps={SEQ_LEN}, features=1)")

In [ ]:
# Build LSTM model untuk time series
tf.random.set_seed(42)

model_lstm_ts = keras.Sequential([
    layers.LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
], name='LSTM_TimeSeries')

model_lstm_ts.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model_lstm_ts.summary()

# Train
print("\nMelatih LSTM untuk time series prediction...")
history_ts = model_lstm_ts.fit(
    X_train_ts, y_train_ts,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=1
)

In [ ]:
# Prediksi dan evaluasi
y_pred_ts = model_lstm_ts.predict(X_test_ts, verbose=0).flatten()

# Inverse transform
y_pred_orig = scaler_ts.inverse_transform(y_pred_ts.reshape(-1, 1)).flatten()
y_test_orig = scaler_ts.inverse_transform(y_test_ts.reshape(-1, 1)).flatten()

mse = np.mean((y_pred_orig - y_test_orig)**2)
mae = np.mean(np.abs(y_pred_orig - y_test_orig))
print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}")

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Prediksi Time Series dengan LSTM', fontsize=13, fontweight='bold')

# Plot prediksi vs aktual
n_plot = 200  # Tampilkan 200 titik
axes[0].plot(y_test_orig[:n_plot], 'b-', linewidth=1.5, label='Aktual', alpha=0.8)
axes[0].plot(y_pred_orig[:n_plot], 'r--', linewidth=1.5, label='Prediksi', alpha=0.8)
axes[0].set_title('Prediksi vs Aktual (200 titik pertama)')
axes[0].set_xlabel('Timestep')
axes[0].set_ylabel('Nilai')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Training loss
axes[1].plot(history_ts.history['loss'], label='Train Loss')
axes[1].plot(history_ts.history['val_loss'], label='Val Loss')
axes[1].set_title('Training Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Aplikasi 2: Analisis Sentimen dengan LSTM

In [ ]:
# Load dataset IMDB Movie Review
MAX_FEATURES = 10000  # Gunakan 10,000 kata terbanyak
MAX_LEN = 200         # Panjang maksimum review

print("Memuat dataset IMDB...")
(X_train_imdb, y_train_imdb), (X_test_imdb, y_test_imdb) = keras.datasets.imdb.load_data(
    num_words=MAX_FEATURES
)

print(f"Training samples: {len(X_train_imdb)}")
print(f"Test samples    : {len(X_test_imdb)}")
print(f"Label: 0=Negatif, 1=Positif")
print(f"\nContoh review (encoded): {X_train_imdb[0][:20]}...")
print(f"Panjang review pertama: {len(X_train_imdb[0])} kata")

# Pad sequences agar panjangnya seragam
X_train_padded = keras.preprocessing.sequence.pad_sequences(X_train_imdb, maxlen=MAX_LEN)
X_test_padded = keras.preprocessing.sequence.pad_sequences(X_test_imdb, maxlen=MAX_LEN)

print(f"\nSetelah padding - shape: {X_train_padded.shape}")

# Decode review untuk ditampilkan
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {v+3: k for k, v in word_index.items()}
reverse_word_index[0] = '[PAD]'
reverse_word_index[1] = '[START]'
reverse_word_index[2] = '[UNK]'

def decode_review(encoded):
    return ' '.join([reverse_word_index.get(i, '?') for i in encoded])

print("\nContoh review yang telah di-decode:")
print(decode_review(X_train_imdb[0][:50]))

In [ ]:
# Build LSTM model untuk sentiment analysis
EMBED_DIM = 64

tf.random.set_seed(42)

model_sentiment = keras.Sequential([
    # Embedding layer: ubah integer index menjadi dense vector
    layers.Embedding(MAX_FEATURES, EMBED_DIM, input_length=MAX_LEN),
    layers.SpatialDropout1D(0.2),
    
    # Bidirectional LSTM: proses sequence dari dua arah
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Dropout(0.3),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.3),
    
    # Classifier
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='SentimentAnalysis_BiLSTM')

model_sentiment.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_sentiment.summary()

print("\nMelatih model analisis sentimen...")
history_sentiment = model_sentiment.fit(
    X_train_padded, y_train_imdb,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

In [ ]:
# Evaluasi
test_loss, test_acc = model_sentiment.evaluate(X_test_padded, y_test_imdb, verbose=0)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

# Visualisasi training
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Hasil Training BiLSTM - Analisis Sentimen IMDB', fontsize=12, fontweight='bold')

axes[0].plot(history_sentiment.history['loss'], label='Train')
axes[0].plot(history_sentiment.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_sentiment.history['accuracy'], label='Train')
axes[1].plot(history_sentiment.history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Prediksi beberapa review
print("\nContoh Prediksi Sentimen:")
print("=" * 60)
sample_indices = [0, 1, 10, 25]  # Beberapa sample test
for idx in sample_indices:
    review_encoded = X_test_padded[idx:idx+1]
    prob = model_sentiment.predict(review_encoded, verbose=0)[0][0]
    label = y_test_imdb[idx]
    pred = 'Positif 😊' if prob > 0.5 else 'Negatif 😞'
    actual = 'Positif' if label == 1 else 'Negatif'
    print(f"Prediksi: {pred} ({prob:.3f}) | Aktual: {actual} | {'✓' if (prob>0.5)==label else '✗'}")

## 6. Perbandingan RNN, LSTM, dan GRU

| Aspek | Simple RNN | LSTM | GRU |
|-------|-----------|------|-----|
| Gate | Tidak ada | 3 gate (forget, input, output) | 2 gate (reset, update) |
| Cell State | Tidak ada | Ada | Tidak ada (digabung dengan h) |
| Parameter | Sedikit | Banyak | Sedang |
| Long-term dependency | Buruk | Baik | Baik |
| Kecepatan training | Cepat | Lambat | Sedang |
| Penggunaan umum | Sequence pendek | NLP, time series | Alternatif LSTM yang lebih ringan |

## 7. Latihan Mandiri

1. **Latihan 1**: Ganti LSTM dengan GRU (`layers.GRU`). Bandingkan performa dan kecepatan training.

2. **Latihan 2**: Tambah lebih banyak LSTM layer. Apakah model mengalami overfitting?

3. **Latihan 3**: Untuk time series, coba prediksi **multi-step** (beberapa langkah ke depan).

4. **Latihan 4**: Gunakan data time series nyata seperti harga saham (yfinance library).

## Ringkasan

✅ **RNN**: Neural network dengan koneksi berulang untuk data sekuensial

✅ **Vanishing Gradient**: Masalah utama RNN untuk sequence panjang

✅ **LSTM**: Solusi vanishing gradient dengan 3 gate (forget, input, output)

✅ **GRU**: Alternatif LSTM yang lebih ringan dengan 2 gate

✅ **Bidirectional LSTM**: Memproses sequence dari dua arah

✅ **Aplikasi**: Prediksi time series dan analisis sentimen

---

**Pertemuan Berikutnya:** Transfer Learning & Pretrained Models 🔄